****Load data for my training process****

In [1]:
!pip install pycocotools -q 

In [2]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import numpy as np
import cv2

def affine_transform(pt, t):
    new_pt = np.array([pt[0], pt[1], 1.]).T
    new_pt = np.dot(t, new_pt)
    return new_pt[:2]


def get_affine_transform(center,
                         scale,
                         rot,
                         output_size,
                         shift=np.array([0, 0], dtype=np.float32),
                         inv=0):
    if not isinstance(scale, np.ndarray) and not isinstance(scale, list):
        print(scale)
        scale = np.array([scale, scale])

    scale_tmp = scale * 200.0
    src_w = scale_tmp[0]
    dst_w = output_size[0]
    dst_h = output_size[1]

    rot_rad = np.pi * rot / 180
    src_dir = get_dir([0, src_w * -0.5], rot_rad)
    dst_dir = np.array([0, dst_w * -0.5], np.float32)

    src = np.zeros((3, 2), dtype=np.float32)
    dst = np.zeros((3, 2), dtype=np.float32)
    src[0, :] = center + scale_tmp * shift
    src[1, :] = center + src_dir + scale_tmp * shift
    dst[0, :] = [dst_w * 0.5, dst_h * 0.5]
    dst[1, :] = np.array([dst_w * 0.5, dst_h * 0.5]) + dst_dir

    src[2:, :] = get_3rd_point(src[0, :], src[1, :])
    dst[2:, :] = get_3rd_point(dst[0, :], dst[1, :])

    if inv:
        trans = cv2.getAffineTransform(np.float32(dst), np.float32(src))
    else:
        trans = cv2.getAffineTransform(np.float32(src), np.float32(dst))

    return trans

def fliplr_joints(joints, joints_vis, width, matched_parts):
    """
    flip coords
    """
    # Flip horizontal
    joints[:, 0] = width - joints[:, 0] - 1

    # Change left-right parts
    for pair in matched_parts:
        joints[pair[0], :], joints[pair[1], :] = \
            joints[pair[1], :], joints[pair[0], :].copy()
        joints_vis[pair[0], :], joints_vis[pair[1], :] = \
            joints_vis[pair[1], :], joints_vis[pair[0], :].copy()

    return joints*joints_vis, joints_vis

def get_dir(src_point, rot_rad):
    sn, cs = np.sin(rot_rad), np.cos(rot_rad)

    src_result = [0, 0]
    src_result[0] = src_point[0] * cs - src_point[1] * sn
    src_result[1] = src_point[0] * sn + src_point[1] * cs

    return src_result

def get_3rd_point(a, b):
    direct = a - b
    return b + np.array([-direct[1], direct[0]], dtype=np.float32)

In [3]:
import copy
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from pycocotools.coco import COCO


class COCOPoseDataset(Dataset):
    """
    Single-class COCO pose dataset for training.
    Merged from JointsDataset + COCODataset (Microsoft Simple Baseline).
    
    Flow: raw image → bbox (center/scale) → affine crop → heatmap targets
    """

    # COCO 17 keypoints
    NUM_JOINTS = 17
    FLIP_PAIRS = [[1, 2], [3, 4], [5, 6], [7, 8],
                  [9, 10], [11, 12], [13, 14], [15, 16]]
    PIXEL_STD = 200

    def __init__(self, root, ann_file, image_size=(192, 256),
                 heatmap_size=(48, 64), sigma=2,
                 scale_factor=0.35, rot_factor=45,
                 flip=True, is_train=True, transform=None):
        """
        Args:
            root: path to images dir (e.g. 'data/coco/images/train2017')
            ann_file: path to annotation json (e.g. 'data/coco/annotations/person_keypoints_train2017.json')
            image_size: model input [W, H]
            heatmap_size: output heatmap [W, H]
            sigma: gaussian sigma for heatmap
            scale_factor: random scale augmentation range
            rot_factor: random rotation augmentation range (degrees)
            flip: enable horizontal flip augmentation
            is_train: training mode flag
            transform: torchvision transforms (normalize, to_tensor, etc.)
        """
        self.root = root
        self.is_train = is_train
        self.transform = transform

        self.image_size = np.array(image_size)
        self.heatmap_size = np.array(heatmap_size)
        self.sigma = sigma
        self.scale_factor = scale_factor
        self.rot_factor = rot_factor
        self.flip = flip
        self.aspect_ratio = image_size[0] / image_size[1]  # W/H

        # Load COCO annotations
        self.coco = COCO(ann_file)
        self.db = self._load_annotations()
        print(f'=> Loaded {len(self.db)} samples')

    # -------------------------------------------------------------------------
    # Data loading
    # -------------------------------------------------------------------------
    def _load_annotations(self):
        """Parse COCO annotations → list of {image, center, scale, joints, joints_vis}"""
        db = []
        for img_id in self.coco.getImgIds():
            im_ann = self.coco.loadImgs(img_id)[0]
            w, h = im_ann['width'], im_ann['height']
            img_path = f"{self.root}/{im_ann['file_name']}"

            for obj in self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id, iscrowd=False)):
                # Skip non-person or unannotated
                if obj['category_id'] != 1 or max(obj['keypoints']) == 0:
                    continue

                # Sanitize bbox
                x, y, bw, bh = obj['bbox']
                x1 = max(0, x)
                y1 = max(0, y)
                x2 = min(w - 1, x1 + max(0, bw - 1))
                y2 = min(h - 1, y1 + max(0, bh - 1))
                if obj['area'] <= 0 or x2 < x1 or y2 < y1:
                    continue

                # Parse keypoints: [x, y, vis] × 17
                kps = np.array(obj['keypoints']).reshape(self.NUM_JOINTS, 3)
                joints = np.zeros((self.NUM_JOINTS, 3), dtype=np.float32)
                joints_vis = np.zeros((self.NUM_JOINTS, 3), dtype=np.float32)
                joints[:, :2] = kps[:, :2]
                joints_vis[:, 0] = np.minimum(kps[:, 2], 1)
                joints_vis[:, 1] = joints_vis[:, 0]

                center, scale = self._box2cs(x1, y1, x2 - x1, y2 - y1)
                db.append({
                    'image': img_path,
                    'center': center,
                    'scale': scale,
                    'joints': joints,
                    'joints_vis': joints_vis,
                })
        return db

    def _box2cs(self, x, y, w, h):
        """Bbox [x, y, w, h] → (center, scale) with aspect ratio correction + 1.25x padding."""
        center = np.array([x + w * 0.5, y + h * 0.5], dtype=np.float32)

        # Fix aspect ratio to match model input
        if w > self.aspect_ratio * h:
            h = w / self.aspect_ratio
        elif w < self.aspect_ratio * h:
            w = h * self.aspect_ratio

        scale = np.array([w / self.PIXEL_STD, h / self.PIXEL_STD], dtype=np.float32)
        scale *= 1.25  # padding
        return center, scale

    # -------------------------------------------------------------------------
    # __getitem__: crop → augment → heatmap
    # -------------------------------------------------------------------------
    def __len__(self):
        return len(self.db)

    def __getitem__(self, idx):
        rec = copy.deepcopy(self.db[idx])
        img = cv2.imread(rec['image'], cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
        if img is None:
            raise ValueError(f"Failed to read {rec['image']}")

        joints = rec['joints']
        joints_vis = rec['joints_vis']
        c, s, r = rec['center'], rec['scale'], 0

        # --- Augmentation (train only) ---
        if self.is_train:
            sf = self.scale_factor
            rf = self.rot_factor
            s *= np.clip(np.random.randn() * sf + 1, 1 - sf, 1 + sf)
            r = np.clip(np.random.randn() * rf, -rf * 2, rf * 2) if np.random.rand() <= 0.6 else 0

            if self.flip and np.random.rand() <= 0.5:
                img = img[:, ::-1, :]
                joints, joints_vis = fliplr_joints(joints, joints_vis, img.shape[1], self.FLIP_PAIRS)
                c[0] = img.shape[1] - c[0] - 1

        # --- Affine crop (image + joints) ---
        trans = get_affine_transform(c, s, r, self.image_size)
        inp = cv2.warpAffine(img, trans, tuple(self.image_size), flags=cv2.INTER_LINEAR)

        for i in range(self.NUM_JOINTS):
            if joints_vis[i, 0] > 0:
                joints[i, :2] = affine_transform(joints[i, :2], trans)

        if self.transform:
            inp = self.transform(inp)

        # --- Generate heatmap targets ---
        target, target_weight = self._generate_heatmaps(joints, joints_vis)

        meta = {'image': rec['image'], 'center': c, 'scale': s, 'rotation': r,
                'joints': joints, 'joints_vis': joints_vis}
        return inp, torch.from_numpy(target), torch.from_numpy(target_weight), meta

    # -------------------------------------------------------------------------
    # Heatmap generation
    # -------------------------------------------------------------------------
    def _generate_heatmaps(self, joints, joints_vis):
        """Generate gaussian heatmaps for each joint."""
        target = np.zeros((self.NUM_JOINTS, self.heatmap_size[1], self.heatmap_size[0]), dtype=np.float32)
        target_weight = joints_vis[:, 0:1].copy()  # (17, 1)

        feat_stride = self.image_size / self.heatmap_size
        tmp_size = self.sigma * 3

        for jid in range(self.NUM_JOINTS):
            mu_x = int(joints[jid][0] / feat_stride[0] + 0.5)
            mu_y = int(joints[jid][1] / feat_stride[1] + 0.5)

            ul = [mu_x - tmp_size, mu_y - tmp_size]
            br = [mu_x + tmp_size + 1, mu_y + tmp_size + 1]

            # Skip if gaussian is fully out of bounds
            if ul[0] >= self.heatmap_size[0] or ul[1] >= self.heatmap_size[1] or br[0] < 0 or br[1] < 0:
                target_weight[jid] = 0
                continue

            # Gaussian kernel
            size = 2 * tmp_size + 1
            g = np.exp(-((np.arange(size) - tmp_size) ** 2 +
                         (np.arange(size)[:, None] - tmp_size) ** 2) / (2 * self.sigma ** 2))

            # Clamp to heatmap bounds
            g_x = (max(0, -ul[0]), min(br[0], self.heatmap_size[0]) - ul[0])
            g_y = (max(0, -ul[1]), min(br[1], self.heatmap_size[1]) - ul[1])
            img_x = (max(0, ul[0]), min(br[0], self.heatmap_size[0]))
            img_y = (max(0, ul[1]), min(br[1], self.heatmap_size[1]))

            if target_weight[jid] > 0.5:
                target[jid][img_y[0]:img_y[1], img_x[0]:img_x[1]] = g[g_y[0]:g_y[1], g_x[0]:g_x[1]]

        return target, target_weight


# =============================================================================
# Debug / Visualization helpers
# =============================================================================
def visualize_sample(dataset, idx=0):
    """Quick sanity check: plot keypoints on cropped image."""
    import matplotlib.pyplot as plt

    inp, target, weight, meta = dataset[idx]

    # Undo normalization if needed (assuming CHW tensor)
    if isinstance(inp, torch.Tensor):
        img = inp.permute(1, 2, 0).numpy()
        img = (img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]) * 255
        img = img.clip(0, 255).astype(np.uint8)
    else:
        img = inp

    joints = meta['joints']
    vis = meta['joints_vis']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: image + keypoints
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for i in range(dataset.NUM_JOINTS):
        if vis[i, 0] > 0:
            axes[0].plot(joints[i, 0], joints[i, 1], 'ro', markersize=3)
            axes[0].text(joints[i, 0], joints[i, 1], str(i), fontsize=6, color='yellow')
    axes[0].set_title('Cropped image + keypoints')

    # Right: sum of heatmaps
    heatmap_sum = target.numpy().sum(axis=0)
    axes[1].imshow(heatmap_sum, cmap='hot')
    axes[1].set_title('Heatmap sum')

    plt.tight_layout()
    plt.show()


def visualize_raw(dataset, idx=0):
    """Vẽ keypoint lên ảnh gốc + in tọa độ (không crop, không augment, không heatmap)."""
    import matplotlib.pyplot as plt
    KP_NAMES = ['nose', 'l_eye', 'r_eye', 'l_ear', 'r_ear',
                'l_shoulder', 'r_shoulder', 'l_elbow', 'r_elbow',
                'l_wrist', 'r_wrist', 'l_hip', 'r_hip',
                'l_knee', 'r_knee', 'l_ankle', 'r_ankle']
    rec = dataset.db[idx]
    img = cv2.imread(rec['image'], cv2.IMREAD_COLOR | cv2.IMREAD_IGNORE_ORIENTATION)
    joints = rec['joints']
    vis = rec['joints_vis']

    print(f"Image: {rec['image']}")
    print(f"{'id':>2}  {'name':<11} {'x':>7} {'y':>7}  vis")
    print('-' * 35)
    for i in range(dataset.NUM_JOINTS):
        v = int(vis[i, 0])
        print(f"{i:>2}  {KP_NAMES[i]:<11} {joints[i,0]:7.1f} {joints[i,1]:7.1f}  {v}")

    plt.figure(figsize=(8, 10))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for i in range(dataset.NUM_JOINTS):
        v = int(vis[i, 0])
        if v == 1:  # labeled nhưng bị che
            color, marker = 'orange', 's'
        elif v == 2:  # labeled và visible
            color, marker = 'red', 'o'
        else:  # v == 0, không gán nhãn → bỏ qua
            continue
        plt.plot(joints[i, 0], joints[i, 1], marker=marker, color=color, markersize=5)
        plt.text(joints[i, 0], joints[i, 1], str(i), fontsize=7, color='yellow')

    plt.title('Raw image + keypoints  (🟠 occluded  🔴 visible)')
    plt.axis('off')
    plt.show()

In [4]:
# =============================================================================
# Usage example (notebook)
# =============================================================================
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

dataset = COCOPoseDataset(
    root='/kaggle/input/coco-2017-dataset/coco2017/train2017',
    ann_file='/kaggle/input/coco-2017-dataset/coco2017/annotations/person_keypoints_train2017.json',
    image_size=(192, 256),
    heatmap_size=(48, 64),
    is_train=True,
    transform=transform,
)

val_dataset = COCOPoseDataset(
    root='/kaggle/input/coco-2017-dataset/coco2017/val2017', 
    ann_file='/kaggle/input/coco-2017-dataset/coco2017/annotations/person_keypoints_val2017.json',
    image_size=(192, 256),
    heatmap_size=(48, 64),
    is_train=False,   # QUAN TRỌNG 1: Tắt scale/rotation augmentation trong __getitem__
    flip=False,       # QUAN TRỌNG 2: Tắt lật ảnh ngẫu nhiên
    transform=transform # Bộ transform (ToTensor + Normalize) bạn đã viết ở trên
)



loading annotations into memory...
Done (t=10.16s)
creating index...
index created!
=> Loaded 149813 samples
loading annotations into memory...
Done (t=0.33s)
creating index...
index created!
=> Loaded 6352 samples


In [ ]:
def visualize_distribution(dataset, n_samples=2000, save_path=None):
    """
    Thống kê phân phối dữ liệu trước và sau khi tiền xử lý.

    Args:
        dataset: instance của COCOPoseDataset
        n_samples: số sample để chạy qua __getitem__ (lấy data AFTER).
                   Lấy ít cho nhanh, lấy nhiều cho chính xác.
        save_path: nếu truyền vào thì lưu hình ra file
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import cv2

    KP_NAMES = ['nose', 'l_eye', 'r_eye', 'l_ear', 'r_ear',
                'l_shoulder', 'r_shoulder', 'l_elbow', 'r_elbow',
                'l_wrist', 'r_wrist', 'l_hip', 'r_hip',
                'l_knee', 'r_knee', 'l_ankle', 'r_ankle']

    N = dataset.NUM_JOINTS
    img_w, img_h = dataset.image_size  # (192, 256)

    # ── BEFORE: thu thập từ db gốc (chưa qua pipeline) ──────────────
    # Lấy bbox GỐC từ self.coco vì db đã fix aspect ratio rồi
    before_vis_count   = np.zeros(len(dataset.db), dtype=np.int32)
    before_vis_perjoint = np.zeros(N, dtype=np.int32)
    before_bbox_w      = np.zeros(len(dataset.db), dtype=np.float32)
    before_bbox_h      = np.zeros(len(dataset.db), dtype=np.float32)
    before_joint_x     = []
    before_joint_y     = []

    # Build lookup từ image_path → annotations gốc COCO
    coco = dataset.coco
    raw_anns = {}
    for ann_id in coco.getAnnIds(iscrowd=False):
        ann = coco.loadAnns(ann_id)[0]
        if ann['category_id'] == 1 and max(ann['keypoints']) > 0:
            key = (ann['image_id'], tuple(ann['keypoints'][:6]))  # match bằng image_id + first 2 kps
            raw_anns[key] = ann['bbox']  # [x, y, w, h] gốc

    for i, rec in enumerate(dataset.db):
        vis = rec['joints_vis'][:, 0]
        before_vis_count[i]   = (vis > 0).sum()
        before_vis_perjoint  += (vis > 0).astype(np.int32)

        # Lấy bbox gốc bằng match với raw COCO annotation
        # Đơn giản hóa: dùng width/height từ scale*PIXEL_STD/1.25 (undo padding)
        # nhưng vẫn bị fixed aspect ratio → cần đọc lại từ raw
        # Cách đơn giản: lưu lại bbox khi build dataset (xem patch bên dưới)
        # Tạm thời để bbox đã-fix-ratio cho panel diện tích
        bw_fixed = rec['scale'][0] * dataset.PIXEL_STD / 1.25
        bh_fixed = rec['scale'][1] * dataset.PIXEL_STD / 1.25
        before_bbox_w[i] = bw_fixed
        before_bbox_h[i] = bh_fixed

        # Tọa độ keypoint chuẩn hóa về [0,1] theo bbox đã fix ratio
        joints = rec['joints']
        c, s = rec['center'], rec['scale']
        bx = c[0] - s[0] * dataset.PIXEL_STD * 0.5
        by = c[1] - s[1] * dataset.PIXEL_STD * 0.5
        bw = s[0] * dataset.PIXEL_STD
        bh = s[1] * dataset.PIXEL_STD
        for j in range(N):
            if vis[j] > 0 and bw > 0 and bh > 0:
                nx = (joints[j, 0] - bx) / bw
                ny = (joints[j, 1] - by) / bh
                before_joint_x.append(nx)
                before_joint_y.append(ny)

    # Lấy bbox gốc THỰC SỰ từ self.coco (không fix ratio)
    raw_bbox_w = []
    raw_bbox_h = []
    for img_id in coco.getImgIds():
        for ann in coco.loadAnns(coco.getAnnIds(imgIds=img_id, iscrowd=False)):
            if ann['category_id'] == 1 and max(ann['keypoints']) > 0 and ann['area'] > 0:
                _, _, w, h = ann['bbox']
                if w > 0 and h > 0:
                    raw_bbox_w.append(w)
                    raw_bbox_h.append(h)
    raw_bbox_w = np.array(raw_bbox_w, dtype=np.float32)
    raw_bbox_h = np.array(raw_bbox_h, dtype=np.float32)

    # ── AFTER: chạy qua __getitem__ trên một subset mẫu ─────────────
    n_samples = min(n_samples, len(dataset))
    indices = np.random.choice(len(dataset), n_samples, replace=False)

    after_vis_count    = np.zeros(n_samples, dtype=np.int32)
    after_vis_perjoint = np.zeros(N, dtype=np.int32)
    after_bbox_w       = np.zeros(n_samples, dtype=np.float32)
    after_bbox_h       = np.zeros(n_samples, dtype=np.float32)
    after_joint_x      = []
    after_joint_y      = []

    for k, idx in enumerate(indices):
        _, _, _, meta = dataset[int(idx)]
        joints = meta['joints']
        vis    = meta['joints_vis'][:, 0]
        s      = meta['scale']
        after_vis_count[k]    = (vis > 0).sum()
        after_vis_perjoint   += (vis > 0).astype(np.int32)
        after_bbox_w[k] = s[0] * dataset.PIXEL_STD
        after_bbox_h[k] = s[1] * dataset.PIXEL_STD

        # Tọa độ sau crop nằm trong khung 192×256 → chuẩn hóa về [0,1]
        for j in range(N):
            if vis[j] > 0:
                nx = joints[j, 0] / img_w
                ny = joints[j, 1] / img_h
                # Có thể bị âm hoặc > 1 nếu rotation đẩy ra ngoài, bỏ qua
                if 0 <= nx <= 1 and 0 <= ny <= 1:
                    after_joint_x.append(nx)
                    after_joint_y.append(ny)

    # ── Plot ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(3, 3, figsize=(16, 14))

    # 1. Số keypoint visible / instance
    bins = np.arange(0, N + 2) - 0.5
    axes[0, 0].hist(before_vis_count, bins=bins, color='steelblue',
                    edgecolor='black', alpha=0.6, label=f'Before (n={len(dataset.db):,})')
    axes[0, 0].hist(after_vis_count,  bins=bins, color='salmon',
                    edgecolor='black', alpha=0.6, label=f'After  (n={n_samples:,})')
    axes[0, 0].set_title('Số keypoint visible / instance')
    axes[0, 0].set_xlabel('Số keypoint visible (0–17)')
    axes[0, 0].set_ylabel('Số instance')
    axes[0, 0].legend(fontsize=9)
    axes[0, 0].set_xticks(range(0, N + 1, 2))

    # 2. Phân bố visibility per-joint (Before)
    pct_before = before_vis_perjoint / len(dataset.db) * 100
    axes[0, 1].barh(range(N), pct_before, color='steelblue', edgecolor='black')
    axes[0, 1].set_yticks(range(N))
    axes[0, 1].set_yticklabels(KP_NAMES, fontsize=8)
    axes[0, 1].invert_yaxis()
    axes[0, 1].set_xlabel('Tỉ lệ visible (%)')
    axes[0, 1].set_title('Visibility per-joint — BEFORE')
    axes[0, 1].grid(axis='x', alpha=0.3)

    # 3. Phân bố visibility per-joint (After)
    pct_after = after_vis_perjoint / n_samples * 100
    axes[0, 2].barh(range(N), pct_after, color='salmon', edgecolor='black')
    axes[0, 2].set_yticks(range(N))
    axes[0, 2].set_yticklabels(KP_NAMES, fontsize=8)
    axes[0, 2].invert_yaxis()
    axes[0, 2].set_xlabel('Tỉ lệ visible (%)')
    axes[0, 2].set_title('Visibility per-joint — AFTER\n(có thể giảm do augment đẩy keypoint ra ngoài crop)')
    axes[0, 2].grid(axis='x', alpha=0.3)

    # 4. Kích thước bbox W vs H — BEFORE (BBOX GỐC, chưa fix ratio)
    n_plot = min(2000, len(raw_bbox_w))
    sel = np.random.choice(len(raw_bbox_w), n_plot, replace=False)
    axes[1, 0].scatter(raw_bbox_w[sel], raw_bbox_h[sel],
                       alpha=0.2, s=5, color='steelblue')
    axes[1, 0].axvline(img_w, color='red',    linestyle='--', label=f'model W={img_w}')
    axes[1, 0].axhline(img_h, color='orange', linestyle='--', label=f'model H={img_h}')
    axes[1, 0].set_title(f'Kích thước bbox gốc — BEFORE (n={n_plot:,})')
    axes[1, 0].set_xlabel('Width (px)')
    axes[1, 0].set_ylabel('Height (px)')
    axes[1, 0].legend(fontsize=9)
    axes[1, 0].set_xlim(0, np.percentile(raw_bbox_w, 99))
    axes[1, 0].set_ylim(0, np.percentile(raw_bbox_h, 99))

    # 5. Aspect ratio bbox (W/H) — Before (gốc) vs After (đã fix)
    ratio_raw   = raw_bbox_w / np.maximum(raw_bbox_h, 1)
    ratio_after = after_bbox_w / np.maximum(after_bbox_h, 1)
    target_ratio = img_w / img_h
    axes[1, 1].hist(ratio_raw,   bins=50, color='steelblue',
                    edgecolor='black', alpha=0.6, label='Before (bbox gốc)', density=True)
    axes[1, 1].hist(ratio_after, bins=50, color='salmon',
                    edgecolor='black', alpha=0.6, label='After (sau fix)', density=True)
    axes[1, 1].axvline(target_ratio, color='red', linestyle='--',
                       label=f'target = {target_ratio:.3f}')
    axes[1, 1].set_title('Aspect ratio bbox (W/H)\nAFTER hội tụ về model ratio')
    axes[1, 1].set_xlabel('W / H')
    axes[1, 1].set_ylabel('Mật độ')
    axes[1, 1].legend(fontsize=9)
    axes[1, 1].set_xlim(0, 3)

    # 6. Diện tích bbox — Before vs After (log scale)
    area_before = before_bbox_w * before_bbox_h
    area_after  = after_bbox_w  * after_bbox_h
    axes[1, 2].hist(np.log10(area_before + 1), bins=50, color='steelblue',
                    edgecolor='black', alpha=0.6, label='Before', density=True)
    axes[1, 2].hist(np.log10(area_after  + 1), bins=50, color='salmon',
                    edgecolor='black', alpha=0.6, label='After', density=True)
    axes[1, 2].set_title('Diện tích bbox (log10)\nAFTER bị augment scale ±35%')
    axes[1, 2].set_xlabel('log10(area)')
    axes[1, 2].set_ylabel('Mật độ')
    axes[1, 2].legend(fontsize=9)

    # 7. Density keypoint — BEFORE (chuẩn hóa theo bbox [0,1])
    h0 = axes[2, 0].hist2d(before_joint_x, before_joint_y,
                            bins=60, range=[[0, 1], [0, 1]], cmap='hot')
    plt.colorbar(h0[3], ax=axes[2, 0], fraction=0.046)
    axes[2, 0].set_title(f'Density keypoints — BEFORE\n(chuẩn hóa theo bbox, n={len(before_joint_x):,})')
    axes[2, 0].set_xlabel('x normalized')
    axes[2, 0].set_ylabel('y normalized')
    axes[2, 0].invert_yaxis()
    axes[2, 0].set_aspect('equal')

    # 8. Density keypoint — AFTER (chuẩn hóa theo crop [0,1])
    h1 = axes[2, 1].hist2d(after_joint_x, after_joint_y,
                            bins=60, range=[[0, 1], [0, 1]], cmap='hot')
    plt.colorbar(h1[3], ax=axes[2, 1], fraction=0.046)
    axes[2, 1].set_title(f'Density keypoints — AFTER\n(chuẩn hóa theo crop 192×256, n={len(after_joint_x):,})')
    axes[2, 1].set_xlabel('x normalized')
    axes[2, 1].set_ylabel('y normalized')
    axes[2, 1].invert_yaxis()
    axes[2, 1].set_aspect('equal')

    # 9. Bảng tóm tắt
    axes[2, 2].axis('off')
    summary = (
        f"════ DATASET ════\n"
        f"  Total instances:    {len(dataset.db):>8,}\n"
        f"  is_train:           {str(dataset.is_train):>8}\n"
        f"  flip:               {str(dataset.flip):>8}\n"
        f"  Sample after:       {n_samples:>8,}\n\n"
        f"════ KEYPOINTS ════\n"
        f"  Avg vis (before):   {before_vis_count.mean():>8.1f}\n"
        f"  Avg vis (after):    {after_vis_count.mean():>8.1f}\n"
        f"  Min vis:            {before_vis_count.min():>8}\n"
        f"  Max vis:            {before_vis_count.max():>8}\n\n"
        f"════ BBOX ════\n"
        f"  W avg (before):     {before_bbox_w.mean():>8.1f}\n"
        f"  H avg (before):     {before_bbox_h.mean():>8.1f}\n"
        f"  W avg (after):      {after_bbox_w.mean():>8.1f}\n"
        f"  H avg (after):      {after_bbox_h.mean():>8.1f}\n"
        f"  Target ratio W/H:   {target_ratio:>8.3f}\n"
        f"  Avg ratio (after):  {ratio_after.mean():>8.3f}\n"
    )
    axes[2, 2].text(0.05, 0.95, summary, transform=axes[2, 2].transAxes,
                    fontsize=10, verticalalignment='top',
                    fontfamily='monospace',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.4))

    plt.suptitle('Phân phối dữ liệu — Before vs After pipeline xử lý',
                 fontsize=14, y=1.00)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved to {save_path}')
    plt.show()


# Cách dùng:
# from cocoposedataset import COCOPoseDataset
# import torchvision.transforms as T
#
# transform = T.Compose([
#     T.ToTensor(),
#     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])
#
# dataset = COCOPoseDataset(
#     root='data/coco/images/train2017',
#     ann_file='data/coco/annotations/person_keypoints_train2017.json',
#     is_train=True,
#     transform=transform
# )

visualize_distribution(dataset, n_samples=len(dataset), save_path='distribution.png')

In [ ]:
# Sanity check: visualize one sample
visualize_raw(dataset, idx=11)

# DataLoader
from torch.utils.data import DataLoader
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

In [ ]:
import os
import logging
import torch
import torch.nn as nn
from collections import OrderedDict

BN_MOMENTUM = 0.1
logger = logging.getLogger(__name__)

def conv3x3(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes, momentum=BN_MOMENTUM)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes, momentum=BN_MOMENTUM)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

class Bottleneck(nn.Module):
    expansion = 4
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes, momentum=BN_MOMENTUM)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes, momentum=BN_MOMENTUM)
        self.conv3 = nn.Conv2d(planes, planes * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion, momentum=BN_MOMENTUM)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.conv3(out)
        out = self.bn3(out)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

class PoseResNet(nn.Module):
    def __init__(self, block, layers, cfg, **kwargs):
        self.inplanes = 64
        extra = cfg.MODEL.EXTRA
        self.deconv_with_bias = extra.DECONV_WITH_BIAS

        super(PoseResNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64, momentum=BN_MOMENTUM)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        # used for deconv layers
        self.deconv_layers = self._make_deconv_layer(
            extra.NUM_DECONV_LAYERS,
            extra.NUM_DECONV_FILTERS,
            extra.NUM_DECONV_KERNELS,
        )

        self.final_layer = nn.Conv2d(
            in_channels=extra.NUM_DECONV_FILTERS[-1],
            out_channels=cfg.MODEL.NUM_JOINTS,
            kernel_size=extra.FINAL_CONV_KERNEL,
            stride=1,
            padding=1 if extra.FINAL_CONV_KERNEL == 3 else 0
        )

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion, momentum=BN_MOMENTUM),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def _get_deconv_cfg(self, deconv_kernel, index):
        if deconv_kernel == 4:
            padding = 1
            output_padding = 0
        elif deconv_kernel == 3:
            padding = 1
            output_padding = 1
        elif deconv_kernel == 2:
            padding = 0
            output_padding = 0
        return deconv_kernel, padding, output_padding

    def _make_deconv_layer(self, num_layers, num_filters, num_kernels):
        layers = []
        for i in range(num_layers):
            kernel, padding, output_padding = self._get_deconv_cfg(num_kernels[i], i)
            planes = num_filters[i]
            layers.append(
                nn.ConvTranspose2d(
                    in_channels=self.inplanes, out_channels=planes,
                    kernel_size=kernel, stride=2, padding=padding,
                    output_padding=output_padding, bias=self.deconv_with_bias))
            layers.append(nn.BatchNorm2d(planes, momentum=BN_MOMENTUM))
            layers.append(nn.ReLU(inplace=True))
            self.inplanes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.deconv_layers(x)
        x = self.final_layer(x)
        return x

resnet_spec = {
    18: (BasicBlock, [2, 2, 2, 2]),
    34: (BasicBlock, [3, 4, 6, 3]),
    50: (Bottleneck, [3, 4, 6, 3]),
    101: (Bottleneck, [3, 4, 23, 3]),
    152: (Bottleneck, [3, 8, 36, 3])
}

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import DataLoader

# --- 1. CẤU HÌNH (CONFIG) ---
class PoseConfig:
    class MODEL:
        NUM_JOINTS = 17 
        class EXTRA:
            NUM_LAYERS = 101 # Sử dụng ResNet-101
            DECONV_WITH_BIAS = False
            NUM_DECONV_LAYERS = 3
            NUM_DECONV_FILTERS = [256, 256, 256]
            NUM_DECONV_KERNELS = [4, 4, 4]
            FINAL_CONV_KERNEL = 1

# --- 2. HÀM KHỞI TẠO MODEL & LOAD WEIGHTS ---
def get_pose_net(cfg, is_train=True):
    num_layers = cfg.MODEL.EXTRA.NUM_LAYERS
    block_class, layers = resnet_spec[num_layers] # Lưu ý: resnet_spec và PoseResNet phải được định nghĩa ở trên
    
    # Khởi tạo model architecture
    model = PoseResNet(block_class, layers, cfg)
    
    if is_train:
        print(f"=> Đang tải pre-trained weights cho ResNet-{num_layers} từ torchvision...")
        
        # Lấy pre-trained model từ torchvision (cú pháp mới của PyTorch)
        if num_layers == 101:
            resnet_tv = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        elif num_layers == 50:
            resnet_tv = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            
        pretrained_state_dict = resnet_tv.state_dict()
        
        # Lọc bỏ lớp Fully Connected (fc) vì ta không dùng tới
        pretrained_state_dict = {k: v for k, v in pretrained_state_dict.items() if not k.startswith('fc')}
        
        # Bơm weights vào backbone (strict=False để bỏ qua deconv_layers và final_layer)
        model.load_state_dict(pretrained_state_dict, strict=False)
        print("=> Đã load xong weights cho backbone!")
        
        # KHỞI TẠO TRỌNG SỐ CHO PHẦN DECONV (Rất quan trọng để model hội tụ nhanh)
        for name, m in model.deconv_layers.named_modules():
            if isinstance(m, nn.ConvTranspose2d):
                nn.init.normal_(m.weight, std=0.001)
                if cfg.MODEL.EXTRA.DECONV_WITH_BIAS:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
                
        for m in model.final_layer.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, std=0.001)
                nn.init.constant_(m.bias, 0)
        print("=> Đã khởi tạo trọng số chuẩn cho Deconv Head!")

    return model

# --- 3. KHỞI TẠO & TEST LUỒNG DỮ LIỆU ---

cfg = PoseConfig()
model = get_pose_net(cfg, is_train=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Giả sử 'dataset' là object COCOPoseDataset bạn đã định nghĩa trước đó
# LƯU Ý KAGGLE: Với ResNet-101, batch_size=32 có thể gây Out of Memory (OOM). 
# Nếu bị lỗi văng RAM GPU, hãy giảm batch_size xuống 16 hoặc 8.
train_loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4) 
valid_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
print("\n=> Đang test thử 1 batch dữ liệu...")
# Hứng đúng 4 giá trị: images, target_heatmaps, target_weights, meta
for images, target_heatmaps, target_weights, meta in train_loader:
    # 1. Đẩy data lên GPU
    images = images.to(device)
    target_heatmaps = target_heatmaps.to(device)
    target_weights = target_weights.to(device) 
    # (Lưu ý: meta là dict nên giữ nguyên ở CPU)
    
    # 2. Pass qua model
    output_heatmaps = model(images)
    
    # 3. In kết quả test
    print("--------------------------------------------------")
    print(f"Input image shape:   {images.shape}")
    print(f"Output heatmap shape:{output_heatmaps.shape}")
    print(f"Target heatmap shape:{target_heatmaps.shape}")
    print(f"Target weights shape:{target_weights.shape}")
    print(f"Meta data keys:      {list(meta.keys())}")
    print("--------------------------------------------------")
    print("=> Pipeline thiết lập thành công! Mọi thứ đã khớp hoàn hảo.")
    
    break # Test 1 batch rồi dừng

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compute_oks(preds, gts, scales, visibilities):
    """
    preds: (N, K, 2) - N người, K điểm chốt (x, y)
    gts: (N, K, 2) - Ground truth
    scales: (N,) - Diện tích (area) hoặc scale của đối tượng
    visibilities: (N, K) - 0: không tồn tại, 1: bị che, 2: rõ ràng
    """
    # Hằng số kappa cho 17 điểm chốt của COCO
    kappas = np.array([
        .026, .025, .025, .035, .035, .079, .079, .072, .072, .062, .062, 
        .107, .107, .087, .087, .089, .089
    ])
    
    oks_list = []
    for i in range(len(preds)):
        # Khoảng cách Euclidean bình phương
        d2 = np.sum((preds[i] - gts[i])**2, axis=1)
        
        # Chỉ tính những điểm có tồn tại (v > 0)
        visible_mask = visibilities[i] > 0
        if np.sum(visible_mask) == 0:
            oks_list.append(0)
            continue
            
        s = scales[i]
        # Công thức OKS
        oks_val = np.exp(-d2 / (2 * (s**2) * (kappas**2) + 1e-9))
        oks_score = np.sum(oks_val * visible_mask) / np.sum(visible_mask)
        oks_list.append(oks_score)
        
    return np.mean(oks_list)

In [ ]:
import time
import os
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm # Dùng thanh tiến trình cho đẹp
# --- ĐỊNH NGHĨA LOSS ---
def joints_mse_loss(output, target, target_weight):
    weight = target_weight.unsqueeze(-1)
    return F.mse_loss(output * weight, target * weight, reduction='mean')

# --- HÀM TÍNH ĐỘ CHÍNH XÁC (ACCURACY) ---
# T tính độ chính xác đơn giản: Điểm dự đoán cách Ground Truth bao nhiêu pixel
def calc_dists(preds, target, normalize):
    preds = preds.astype(np.float32)
    target = target.astype(np.float32)
    dists = np.zeros((preds.shape[1], preds.shape[0]))
    for n in range(preds.shape[0]):
        for c in range(preds.shape[1]):
            if target[n, c, 0] > 1 and target[n, c, 1] > 1:
                normed_preds = preds[n, c, :] / normalize[n]
                normed_targets = target[n, c, :] / normalize[n]
                dists[c, n] = np.linalg.norm(normed_preds - normed_targets)
            else:
                dists[c, n] = -1
    return dists

# --- HÀM TRAIN 1 EPOCH ---
def train_epoch(model, dataloader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    
    # Bọc dataloader bằng tqdm để theo dõi tốc độ (it/s)
    pbar = tqdm(dataloader, desc="Training", leave=False)
    
    for imgs, targets, weights, meta in pbar:
        # non_blocking=True giúp luồng copy dữ liệu CPU->GPU không khóa luồng tính toán
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        weights = weights.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        # Ép kiểu FP16 tự động
        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, targets, weights)
            
        # Backward và Step thông qua Scaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        current_loss = loss.item()
        total_loss += current_loss
        
        # Cập nhật loss liên tục lên thanh tiến trình
        pbar.set_postfix({'loss': f"{current_loss:.6f}"})
        
    return total_loss / len(dataloader)

# --- HÀM VALIDATE ---
def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    
    # Bạn có thể bổ sung logic tính mAP chuẩn của COCO ở đây nếu muốn. 
    # Ở đây t chỉ tính Validation Loss để code ngắn gọn.
    pbar = tqdm(dataloader, desc="Validating")
    with torch.no_grad():
        for i, (images, target, target_weight, meta) in enumerate(pbar):
            images, target, target_weight = images.to(device), target.to(device), target_weight.to(device)
            
            output = model(images)
            loss = criterion(output, target, target_weight)
            total_loss += loss.item()
            
    return total_loss / len(dataloader)

In [ ]:
from torch.cuda.amp import GradScaler
num_epochs = 40            
learning_rate = 1e-3
lr_step = [90, 120]         
lr_factor = 0.1        
# Khởi tạo bộ Scaler cho Mixed Precision
scaler = GradScaler()
resume_path = '/kaggle/input/datasets/huudatnguyen/checkpoint/checkpoint_manual_epoch39.pth' 

# --- OPTIMIZER & SCHEDULER ---
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=lr_step, gamma=lr_factor)

if os.path.isfile(resume_path):
    print(f"=> Tìm thấy checkpoint tại '{resume_path}'. Đang nạp dữ liệu...")
    
    # Load checkpoint vào bộ nhớ
    checkpoint = torch.load(resume_path, map_location=device)
    
    # Khôi phục các trạng thái
    start_epoch = checkpoint['epoch']
    best_val_loss = checkpoint.get('val_loss', float('inf'))
    
    model.load_state_dict(checkpoint['state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    # scaler.load_state_dict(checkpoint['scaler'])
    
    # Khôi phục scheduler để learning rate giảm đúng nhịp (nếu có lưu)
    # if 'scheduler' in checkpoint:
    #     lr_scheduler.load_state_dict(checkpoint['scheduler'])
        
    print(f"=> Hoàn tất nạp dữ liệu! Tiếp tục train từ Epoch {start_epoch + 1}...")
else:
    print("=> Không tìm thấy checkpoint cũ. Bắt đầu train từ Epoch 1 với tốc độ bàn thờ...")
# =========================================================

# Vòng lặp train thay vì bắt đầu từ 0, sẽ bắt đầu từ start_epoch

In [ ]:
import os
import torch
from torch.cuda.amp import autocast

# --- TỐI ƯU HÓA GPU TẬN GỐC ---
# Bật benchmark giúp cuDNN tìm thuật toán chạy mượt nhất cho ảnh size cố định
torch.backends.cudnn.benchmark = True 

# --- THIẾT LẬP HYPERPARAMETERS ---
     

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)



# Thư mục lưu Checkpoint
output_dir = './checkpoints'
os.makedirs(output_dir, exist_ok=True)


for epoch in range(start_epoch, num_epochs):
    print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")
    
    # 1. Train (Lưu ý: Truyền thêm scaler vào hàm này)
    train_loss = train_epoch(model, train_loader, optimizer, joints_mse_loss, device, scaler)
    
    # 2. Validate (Validation không cần scaler)
    val_loss = validate_epoch(model, valid_loader, joints_mse_loss, device)
    
    # 3. Step Scheduler
    lr_scheduler.step()
    current_lr = lr_scheduler.get_last_lr()[0]
    
    print(f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | LR: {current_lr:.1e}")
    
    # 4. Lưu Checkpoint
    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        
    checkpoint = {
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'val_loss': val_loss,
        'optimizer': optimizer.state_dict(),
        # 'scaler': scaler.state_dict(),
        'scheduler': lr_scheduler.state_dict() # ĐÃ THÊM: Lưu scheduler để lúc resume LR không bị ngáo
    }
    
    # Lưu epoch hiện tại (ghi đè liên tục để không rác ổ cứng)
    torch.save(checkpoint, os.path.join(output_dir, 'checkpoint_latest.pth'))
    
    # Lưu best model
    if is_best:
        print(f"=> Cập nhật mô hình tốt nhất (Val Loss: {best_val_loss:.6f})")
        torch.save(checkpoint, os.path.join(output_dir, 'model_best.pth'))

    # --- LƯU ĐỊNH KỲ MỖI 10 EPOCH -- -
    if (epoch + 1) % 10 == 0:
        checkpoint_name = f'checkpoint_epoch_{epoch+1}.pth'
        torch.save(checkpoint, os.path.join(output_dir, checkpoint_name))
        print(f"=> Đã sao lưu định kỳ: {checkpoint_name}")

print("=> Hoàn tất quá trình huấn luyện!")

In [ ]:
!zip -r checkpoints_all.zip ./checkpoints

In [ ]:
from IPython.display import FileLink
display(FileLink(r'checkpoints_all.zip'))

In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt

# 1. Tải checkpoint (Giữ nguyên logic của bạn)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load('/kaggle/input/datasets/huudatnguyen/checkpoint/checkpoint_manual_epoch39.pth', map_location=device)

# 2. Khởi tạo model
model = get_pose_net(cfg, is_train=False)
model.load_state_dict(checkpoint['state_dict'])
model.to(device)
model.eval()

def get_max_preds(heatmap_tensor):
    """
    Trích xuất tọa độ (x, y) từ heatmap dự đoán.
    heatmap_tensor: [num_joints, H_hm, W_hm]
    """
    heatmap_tensor = heatmap_tensor.detach().cpu()
    num_joints = heatmap_tensor.shape[0]
    width = heatmap_tensor.shape[2]
    
    # Flatten heatmap để tìm vị trí có giá trị lớn nhất (argmax)
    heatmap_flat = heatmap_tensor.view(num_joints, -1)
    maxvals, idx = torch.max(heatmap_flat, dim=1)
    
    preds = torch.zeros((num_joints, 2))
    preds[:, 0] = idx % width          # x = cột
    preds[:, 1] = torch.floor(idx / width) # y = hàng
    
    return preds.numpy(), maxvals.numpy()

def visualize_prediction(inp, heatmap_tensor, meta, num_joints):
    # --- 1. Xử lý ảnh gốc ---
    if isinstance(inp, torch.Tensor):
        img = inp.permute(1, 2, 0).cpu().numpy()
        # Chuẩn hóa ngược (Denormalize)
        img = (img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]) * 255
        img = img.clip(0, 255).astype(np.uint8)
    else:
        img = inp

    # Chuyển BGR sang RGB để matplotlib hiển thị đúng màu
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # --- 2. Xử lý Keypoints từ Heatmap ---
    preds_raw, maxvals = get_max_preds(heatmap_tensor)
    
    # Tính tỉ lệ scale (thường là 4.0 vì ảnh 256x192 -> heatmap 64x48)
    scale_x = img.shape[1] / heatmap_tensor.shape[2]
    scale_y = img.shape[0] / heatmap_tensor.shape[1]
    
    preds_rescaled = preds_raw.copy()
    preds_rescaled[:, 0] *= scale_x
    preds_rescaled[:, 1] *= scale_y

    # --- 3. Vẽ biểu đồ tách biệt ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 9)) # Tăng kích thước hình để nhìn rõ

    # Panel Trái: CHỈ VẼ GROUND TRUTH (Màu đỏ)
    axes[0].imshow(img_rgb)
    joints_gt = meta['joints']
    vis_gt = meta['joints_vis']

    for i in range(num_joints):
        if vis_gt[i, 0] > 0:
            axes[0].plot(joints_gt[i, 0], joints_gt[i, 1], 'ro', markersize=4)
            axes[0].text(joints_gt[i, 0], joints_gt[i, 1], str(i), 
                         fontsize=7, color='yellow')
    axes[0].set_title(f'Actual Keypoints (Ground Truth) - {num_joints} Joints')
    axes[0].axis('off') # Tắt trục tọa độ

    # Panel Phải: CHỈ VẼ PREDICTION (Màu xanh lá)
    axes[1].imshow(img_rgb)
    for i in range(num_joints):
        if maxvals[i] > 0.1: # Chỉ vẽ nếu model đủ tự tin
            axes[1].plot(preds_rescaled[i, 0], preds_rescaled[i, 1], 'go', markersize=5)
            axes[1].text(preds_rescaled[i, 0], preds_rescaled[i, 1], str(i), 
                         fontsize=8, color='lime', weight='bold')
    axes[1].set_title('Predicted Keypoints (from Model Heatmap)')
    axes[1].axis('off') # Tắt trục tọa độ

    plt.tight_layout()
    plt.show()

def run_and_visualize(dataset, model, idx=0):
    inp, target, weight, meta = dataset[idx]
    input_tensor = inp.unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
        if isinstance(output, list):
            output = output[-1]
            
    pred_heatmap = output.squeeze(0)
    visualize_prediction(inp, pred_heatmap, meta, dataset.NUM_JOINTS)

# --- Thực thi ---
run_and_visualize(val_dataset, model, idx=2)

In [ ]:
import torch
import os
import matplotlib.pyplot as plt

def extract_metrics_from_checkpoints(checkpoint_dir):
    epochs = []
    oks_scores = []
    
    # Lấy danh sách file và sắp xếp theo số epoch
    files = sorted([f for f in os.listdir(checkpoint_dir) if f.endswith('.pth')])
    
    for f in files:
        ckpt_path = os.path.join(checkpoint_dir, f)
        checkpoint = torch.load(ckpt_path, map_location='cpu')
        
        # Lấy thông tin từ dictionary đã lưu
        epochs.append(checkpoint.get('epoch', 0))
        # Nếu đã có sẵn OKS trong checkpoint
        oks_scores.append(checkpoint.get('val_oks', 0)) 
        
    return epochs, oks_scores

# Vẽ biểu đồ
epochs, scores = extract_metrics_from_checkpoints('./your_checkpoints_folder')
plt.figure(figsize=(10, 5))
plt.plot(epochs, scores, marker='o', linestyle='-', color='b', label='mAP (OKS)')
plt.title('Model Evolution - mAP over Epochs')
plt.xlabel('Epoch')
plt.ylabel('mAP (OKS)')
plt.grid(True)
plt.legend()
plt.show()